In [1]:
import cv2
import numpy as np
from sklearn.cluster import KMeans
from PIL import Image
from rfdetr import RFDETRBase

/Users/jemima/anaconda3/envs/football-detection-env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
full_classes = ['player-ball-goalkeeper-referee-QfA4', 'ball', 'goalkeeper', 'player', 'referee']
class_names  = full_classes[1:]

model = RFDETRBase(pretrain_weights='models/RFDETR Result Dataset With Augmentation Version 1/checkpoint_best_total.pth', num_classes=4)

cap = cv2.VideoCapture('input_video/08fd33_4.mp4')
ret, frame = cap.read()
cap.release()

pil_image  = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
detections = model.predict(pil_image, threshold=0.3)

new_class_ids = []
for raw_id in detections.class_id:
    mapped_id = int(raw_id) - 1
    mapped_id = max(0, min(mapped_id, len(class_names) - 1))
    new_class_ids.append(mapped_id)
detections.class_id = np.array(new_class_ids)

print("=== Dominant Colors Per Player ===")
for i in range(len(detections.xyxy)):
    cls_id   = int(detections.class_id[i])
    cls_name = class_names[cls_id]

    if cls_name not in ['player', 'goalkeeper']:
        continue

    bbox = detections.xyxy[i].tolist()
    x1, y1, x2, y2 = [int(v) for v in bbox]
    w, h = x2 - x1, y2 - y1

    # Center crop — area jersey
    cx, cy   = (x1 + x2) // 2, y1 + int(h * 0.35)
    half_w   = int(w * 0.15)
    half_h   = int(h * 0.15)
    crop     = frame[max(0, cy-half_h):cy+half_h, max(0, cx-half_w):cx+half_w]

    if crop.size == 0:
        continue

    pixels = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB).reshape(-1, 3).astype(np.float32)
    if len(pixels) < 3:
        continue

    km = KMeans(n_clusters=3, n_init=3, random_state=42)
    km.fit(pixels)
    labels, counts = np.unique(km.labels_, return_counts=True)
    top_color = km.cluster_centers_[np.argmax(counts)].astype(int)

    print(f"[{cls_name}] bbox={x1},{y1},{x2},{y2} → top color RGB: {top_color}")

[2026-04-21 22:48:38] [WARNING] rf-detr - Model is not optimized for inference. Latency may be higher than expected. You can optimize the model for inference by calling model.optimize_for_inference().


=== Dominant Colors Per Player ===
[player] bbox=1368,816,1443,902 → top color RGB: [226 236 246]
[player] bbox=583,590,627,672 → top color RGB: [231 244 247]
[player] bbox=533,688,577,787 → top color RGB: [176 253 130]
[player] bbox=1228,431,1263,502 → top color RGB: [230 245 247]
[player] bbox=222,511,253,593 → top color RGB: [246 249 252]
[player] bbox=872,362,902,423 → top color RGB: [190 245 162]
[player] bbox=1310,447,1350,516 → top color RGB: [185 249 164]
[player] bbox=376,306,399,365 → top color RGB: [243 252 251]
[player] bbox=360,722,392,826 → top color RGB: [240 243 249]
[player] bbox=774,418,803,491 → top color RGB: [240 250 250]
[player] bbox=1160,355,1185,412 → top color RGB: [234 240 236]
[player] bbox=836,634,899,721 → top color RGB: [179 247 151]
[player] bbox=1123,705,1171,793 → top color RGB: [211 238 233]
[player] bbox=777,366,798,426 → top color RGB: [34 38 33]
[goalkeeper] bbox=1902,376,1920,442 → top color RGB: [206 130  99]
[player] bbox=949,224,968,274 → top c

In [4]:
detections = model.predict(pil_image, threshold=0.3)  # turunkan dari 0.3 ke 0.1

# Print semua class yang terdeteksi
print("=== Semua Deteksi ===")
for i in range(len(detections.xyxy)):
    cls_id   = int(detections.class_id[i]) - 1
    cls_id   = max(0, min(cls_id, len(class_names) - 1))
    cls_name = class_names[cls_id]
    conf     = detections.confidence[i]
    bbox     = detections.xyxy[i].tolist()
    print(f"[{cls_name}] conf={conf:.2f} bbox={int(bbox[0])},{int(bbox[1])},{int(bbox[2])},{int(bbox[3])}")

=== Semua Deteksi ===
[player] conf=0.94 bbox=1368,816,1443,902
[player] conf=0.92 bbox=583,590,627,672
[player] conf=0.90 bbox=533,688,577,787
[player] conf=0.90 bbox=1228,431,1263,502
[player] conf=0.90 bbox=222,511,253,593
[player] conf=0.89 bbox=872,362,902,423
[player] conf=0.89 bbox=1310,447,1350,516
[player] conf=0.88 bbox=376,306,399,365
[player] conf=0.88 bbox=360,722,392,826
[player] conf=0.88 bbox=774,418,803,491
[player] conf=0.87 bbox=1160,355,1185,412
[player] conf=0.86 bbox=836,634,899,721
[player] conf=0.86 bbox=1123,705,1171,793
[player] conf=0.85 bbox=777,366,798,426
[goalkeeper] conf=0.84 bbox=1902,376,1920,442
[player] conf=0.83 bbox=949,224,968,274
[ball] conf=0.80 bbox=1186,849,1203,865
[player] conf=0.76 bbox=1574,610,1611,695
[referee] conf=0.76 bbox=1095,304,1113,360
[player] conf=0.76 bbox=1280,393,1304,458
[player] conf=0.75 bbox=996,451,1025,527
[referee] conf=0.73 bbox=328,494,363,570
[player] conf=0.62 bbox=1149,714,1202,795
[referee] conf=0.55 bbox=308,22

In [4]:
import torch
ckpt = torch.load('models/RFDETR Result Dataset With Augmentation Version 1/best_model.pt', map_location='cpu')
print(type(ckpt))
if isinstance(ckpt, dict):
    print("Keys:", list(ckpt.keys()))

<class 'collections.OrderedDict'>
Keys: ['transformer.decoder.layers.0.self_attn.in_proj_weight', 'transformer.decoder.layers.0.self_attn.in_proj_bias', 'transformer.decoder.layers.0.self_attn.out_proj.weight', 'transformer.decoder.layers.0.self_attn.out_proj.bias', 'transformer.decoder.layers.0.norm1.weight', 'transformer.decoder.layers.0.norm1.bias', 'transformer.decoder.layers.0.cross_attn.sampling_offsets.weight', 'transformer.decoder.layers.0.cross_attn.sampling_offsets.bias', 'transformer.decoder.layers.0.cross_attn.attention_weights.weight', 'transformer.decoder.layers.0.cross_attn.attention_weights.bias', 'transformer.decoder.layers.0.cross_attn.value_proj.weight', 'transformer.decoder.layers.0.cross_attn.value_proj.bias', 'transformer.decoder.layers.0.cross_attn.output_proj.weight', 'transformer.decoder.layers.0.cross_attn.output_proj.bias', 'transformer.decoder.layers.0.linear1.weight', 'transformer.decoder.layers.0.linear1.bias', 'transformer.decoder.layers.0.linear2.weight'

In [1]:
import torch

# Wrap best_model.pt ke format yang diharapkan RFDETRBase
ckpt_path = 'models/RFDETR Result Dataset With Augmentation Version 1/best_model_new.pt'
wrapped_path = 'models/RFDETR Result Dataset With Augmentation Version 1/best_model_new_wrapped.pt'

state_dict = torch.load(ckpt_path, map_location='cpu')
torch.save({'model': state_dict}, wrapped_path)
print("✅ Wrapped checkpoint saved!")

✅ Wrapped checkpoint saved!


In [6]:
cap = cv2.VideoCapture('input_video/08fd33_4.mp4')
print("FPS:", cap.get(cv2.CAP_PROP_FPS))
cap.release()

FPS: 25.0


In [7]:
import inspect
import supervision as sv
print(inspect.signature(sv.ByteTrack.__init__))

(self, track_activation_threshold: float = 0.25, lost_track_buffer: int = 30, minimum_matching_threshold: float = 0.8, frame_rate: int = 30, minimum_consecutive_frames: int = 1)
